# Week 4 Major Project (Capstone Project)
**Option 1: Heart Disease Prediction**

**Objective:** Build a complete ML pipeline to predict if a patient is likely to have heart disease using the UCI Heart Disease dataset.

## 1. Data Cleaning and Feature Scaling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Load Data from UCI Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv(url, names=columns)

# Data Cleaning
# The dataset contains '?' for missing values in some columns
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

# Convert all columns to numeric types
df = df.apply(pd.to_numeric)

# The target variable ranges from 0 (no presence) to 4 (high presence).
# We convert it to a binary classification problem: 0 = No Disease, 1 = Disease
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# Split features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data cleaning, splitting, and scaling completed successfully.")

## 2. Train Models (Logistic Regression & Decision Tree)

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Train Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train_scaled, y_train)
dt_preds = dt_model.predict(X_test_scaled)
dt_probs = dt_model.predict_proba(X_test_scaled)[:, 1]

print("Models trained successfully.")

## 3. Evaluate Models (Accuracy, Precision, Recall)

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}\n")

evaluate_model(y_test, lr_preds, "Logistic Regression")
evaluate_model(y_test, dt_preds, "Decision Tree")

## 4. Visualizations (Confusion Matrix & ROC Curve)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Confusion Matrix - Logistic Regression
sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Logistic Regression Confusion Matrix')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# Confusion Matrix - Decision Tree
sns.heatmap(confusion_matrix(y_test, dt_preds), annot=True, fmt='d', cmap='Greens', ax=axes[0, 1])
axes[0, 1].set_title('Decision Tree Confusion Matrix')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Actual')

# ROC Curve - Logistic Regression
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
roc_auc_lr = auc(fpr_lr, tpr_lr)
axes[1, 0].plot(fpr_lr, tpr_lr, color='blue', lw=2, label=f'AUC = {roc_auc_lr:.2f}')
axes[1, 0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
axes[1, 0].set_title('Logistic Regression ROC Curve')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].legend(loc="lower right")

# ROC Curve - Decision Tree
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_probs)
roc_auc_dt = auc(fpr_dt, tpr_dt)
axes[1, 1].plot(fpr_dt, tpr_dt, color='green', lw=2, label=f'AUC = {roc_auc_dt:.2f}')
axes[1, 1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
axes[1, 1].set_title('Decision Tree ROC Curve')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## Conclusion
Logistic regression outperforms the basic decision tree on this heart disease dataset, as evidenced by a higher accuracy, better recall, and a greater Area Under the ROC Curve (AUC). For heart disease prediction, higher recall is often prioritized since false negatives are more dangerous.